# P5 numerical pricing and hedge-cost foundation

All saved experiments are synthetic. There is no observed quote chain or trading track record.
This notebook executes ordinary Python sequentially. A Jupyter-kernel run is not claimed.


In [1]:
from pathlib import Path
import json,sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/options_lab').is_dir())
sys.path.insert(0,str(ROOT/'src'))
from options_lab.pricing import bsm,crr,implied_vol,greeks,monte_carlo
from options_lab.hedging import gbm_path,hedge_path
print('Benchmark call:',bsm(100,100,1,.05,.2))


Benchmark call: 10.450583572185565


## 1 Price to IV to price
Solver status and price residual must accompany the implied volatility.

In [2]:
p=bsm(100,100,1,.05,.2)
r=implied_vol(p,100,100,1,.05)
assert r['status']=='CONVERGED' and abs(r['residual'])<1e-9
print(r)
print(greeks(100,100,1,.05,.2))


{'status': 'CONVERGED', 'iv': 0.20000000001164153, 'residual': 4.368274630905944e-10, 'iterations': 33}
{'status': 'SMOOTH', 'delta': 0.6368306511756191, 'gamma': 0.018762017345846895, 'vega_per_vol_point': 0.3752403469169379, 'theta_per_calendar_day': -0.01757267820941972, 'rho_per_rate_point': 0.5323248154537634}


## 2 Numerical approximation and uncertainty
An individual Monte Carlo interval can miss the correct price. Standard errors belong to independent antithetic pairs.

In [3]:
results=json.loads((ROOT/'outputs/research_results.json').read_text())
for r in results['benchmark']['tree']:print('CRR',r)
print('Monte Carlo:',results['benchmark']['monte_carlo'])
print('Coverage diagnostic:',results['monte_carlo_coverage'])


CRR {'steps': 25, 'price': 10.520965623983946, 'error': 0.07038205179838108}
CRR {'steps': 50, 'price': 10.410691540732644, 'error': -0.03989203145292031}
CRR {'steps': 100, 'price': 10.430611662249113, 'error': -0.019971909936451482}
CRR {'steps': 200, 'price': 10.44059125985994, 'error': -0.009992312325625363}
CRR {'steps': 400, 'price': 10.445585841283892, 'error': -0.004997730901672881}
CRR {'steps': 800, 'price': 10.448084314882404, 'error': -0.0024992573031603627}
Monte Carlo: {'price': 10.47157469115445, 'standard_error': 0.051854994558231, 'ci95': [10.369938901820317, 10.573210480488582], 'independent_pairs': 20000, 'terminal_draws': 40000, 'seed': 1701, 'measure': 'risk_neutral', 'interval_method': 'normal approximation on antithetic pair averages'}
Coverage diagnostic: {'replications': 40, 'independent_pairs_per_replication': 2000, 'coverage': 0.925, 'wilson95': [0.8013546984090248, 0.9741644432281419], 'interpretation': 'Monte Carlo coverage diagnostic with binomial uncertai

## 3 Self-financing cash ledger
Premium finances the hedge and a cash account that may borrow. Final stock liquidation and option settlement both enter cash.

In [4]:
p=gbm_path(100,.25,.25,.04,256,94000)
h=hedge_path(p,100,.25,.04,.25,rebalance_every=4,fee_bps=5)
assert abs(h['accounting_residual'])<1e-10 and h['terminal_holdings']==0
print({k:h[k] for k in ['premium','terminal_payoff','total_costs','total_interest','hedging_error','accounting_residual']})


{'premium': 5.472087330311716, 'terminal_payoff': 1.7127931667503304, 'total_costs': 0.2433403506517886, 'total_interest': -0.4871118113482312, 'hedging_error': 0.5434165114284468, 'accounting_residual': -2.375877272697835e-14}


## 4 Cost and frequency use identical paths
A lower discretisation error does not imply a better result after costs. These exploratory results do not select a universal optimum.

In [5]:
matched=[r for r in results['hedge_cases'] if r['realised_vol']==.25]
for r in matched:print('Every',r['rebalance_every'],'steps; fees',r['fee_bps'],'bps; RMSE',round(r['rmse'],4),'mean error',round(r['mean_error'],4))
print('Largest accounting residual:',results['max_cash_accounting_residual'])


Every 1 steps; fees 0 bps; RMSE 0.241 mean error 0.0177
Every 1 steps; fees 5 bps; RMSE 0.3841 mean error -0.2765
Every 1 steps; fees 20 bps; RMSE 1.2498 mean error -1.1593
Every 4 steps; fees 0 bps; RMSE 0.4882 mean error 0.0202
Every 4 steps; fees 5 bps; RMSE 0.5172 mean error -0.154
Every 4 steps; fees 20 bps; RMSE 0.8658 mean error -0.6763
Every 16 steps; fees 0 bps; RMSE 1.0119 mean error 0.0237
Every 16 steps; fees 5 bps; RMSE 1.0212 mean error -0.0888
Every 16 steps; fees 20 bps; RMSE 1.1239 mean error -0.4261
Every 64 steps; fees 0 bps; RMSE 1.9678 mean error -0.1337
Every 64 steps; fees 5 bps; RMSE 1.9809 mean error -0.2144
Every 64 steps; fees 20 bps; RMSE 2.0407 mean error -0.4567
Largest accounting residual: 2.824407374646398e-13


## 5 Separate seeds and paired uncertainty
These are conditional synthetic comparisons, not market OOS. A jump is imposed on every jump-stress path. Intervals are pointwise and select no policy.

In [6]:
extension=json.loads((ROOT/'outputs/holdout_results.json').read_text())
assert extension['paths_disjoint_from_baseline'] and not extension['market_out_of_sample']
for row in extension['paired_comparisons']:
    print(row['scenario'],row['fee_bps'],'bps; RMSE difference',round(row['rmse_difference'],4),'95% interval',[round(v,4) for v in row['rmse_difference_percentile95']])
print('Policy selected:',extension['policy_selected'])


flat_25 5 bps; RMSE difference 0.0729 95% interval [-0.0101, 0.1481]
flat_25 20 bps; RMSE difference -0.4525 95% interval [-0.5439, -0.3674]
low_to_high 5 bps; RMSE difference -0.0368 95% interval [-0.2194, 0.126]
low_to_high 20 bps; RMSE difference -0.5124 95% interval [-0.6995, -0.3399]
high_to_low 5 bps; RMSE difference -0.0338 95% interval [-0.1035, 0.0449]
high_to_low 20 bps; RMSE difference -0.4175 95% interval [-0.5008, -0.3303]
conditional_down_jump 5 bps; RMSE difference -0.1439 95% interval [-0.2625, -0.0276]
conditional_down_jump 20 bps; RMSE difference -0.3822 95% interval [-0.5056, -0.2611]
Policy selected: False


## 6 Claims and next research stage
Market snapshots require contract, timestamp and publication-rights review. Calibration accuracy alone cannot establish hedge performance.

In [7]:
validation=json.loads((ROOT/'outputs/validation.json').read_text())
assert validation['status']=='PASS'
print('Mathematical and accounting tests:',validation)
print('Observed market data admitted:',results['observed_market_data_admitted'])
print('Excel model: outputs/P5_Options_Research_Model.xlsx; independent formula and cross-engine checks saved.')
print('Next: allowed observed market evidence, venue-specific hedge accounting and locked challenger evaluation.')


Mathematical and accounting tests: {'tests': 26, 'failures': 0, 'errors': 0, 'status': 'PASS'}
Observed market data admitted: False
Excel model: outputs/P5_Options_Research_Model.xlsx; independent formula and cross-engine checks saved.
Next: allowed observed market evidence, venue-specific hedge accounting and locked challenger evaluation.
